# TurathiAI — Omnibus ANOVA + Per-Ablation IQA (R1-C5 / R1-C6)

Standalone evaluation notebook. It loads each LoRA checkpoint produced by the training notebook
(`lora_default_best` + every `lora_<ablation>_best`), generates the shared test set, and computes
**CLIP / FID / LPIPS / SSIM per model**, then runs the **omnibus one-way ANOVA with eta-squared**
(plus Friedman + Kendall's W and Holm-corrected pairwise paired t-tests with Cohen's d) over the
per-image CLIP scores.

Outputs written to `OUTPUT_DIR`:
- `ablation_iqa.csv` — per-model CLIP / FID / LPIPS / SSIM (feeds manuscript Table 23)
- `clip_per_image_all.csv` — aligned per-image CLIP for every model (reproducibility)
- `anova_omnibus.json` — F, p, eta-squared (+ Friedman chi2, Kendall's W)
- `pairwise_stats.csv` — every pair: t, p_raw, p_holm, Cohen's d, significant

**Run order:** Environment -> Config -> Mount -> Helpers -> Discover checkpoints -> Generate+score -> Stats -> Save.
Set `QUICK=True` first for a fast smoke test (25 prompts x 1 seed) before the full paper-scale run.

In [ ]:
# --- Environment ---
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
%pip -q install "diffusers>=0.30" "transformers>=4.44" "accelerate>=0.33" "peft>=0.12" \
    safetensors "torchmetrics[image]>=1.4" scipy statsmodels pandas tqdm pillow
print("installs done")

NVIDIA L4, 23034 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.5 MB/s eta 0:00:00
installs done


In [ ]:
# ====================== CONFIG (EDIT ME) ======================
from dataclasses import dataclass
@dataclass
class Cfg:
    base_model: str  = "stabilityai/stable-diffusion-xl-base-1.0"
    vae_model: str   = "madebyollin/sdxl-vae-fp16-fix"
    drive_folder: str= "/content/drive/MyDrive/CHDB_Full"                            # real images + test_prompts.txt
    output_dir: str  = "/content/drive/MyDrive/TurathiAI Revision/Training/outputs"  # holds lora_*_best/
    resolution: int  = 1024
    eval_steps: int  = 25
    guidance: float  = 6.0
    images_per_prompt: int = 4     # seeds per prompt (paper protocol = 4)
    seed: int        = 42
CFG = Cfg()

QUICK = False   # True -> 25 prompts x 1 seed (smoke test). Set False for the full 100 x 4 paper run.
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print("device", DEVICE, "| QUICK", QUICK)

device cuda | QUICK False


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
GEN_ROOT = "/content/eval_gen"; os.makedirs(GEN_ROOT, exist_ok=True)
assert os.path.isdir(CFG.output_dir), f"output_dir not found: {CFG.output_dir}"
print("checkpoints dir:", CFG.output_dir)

Mounted at /content/drive
checkpoints dir: /content/drive/MyDrive/TurathiAI Revision/Training/outputs


In [ ]:
# --- Test prompts + real reference paths ---
import os, glob
PF = os.path.join(CFG.drive_folder, "test_prompts.txt")
if os.path.exists(PF):
    TEST_PROMPTS = [l.strip() for l in open(PF) if l.strip()]
else:
    TEST_PROMPTS = [
        "traditional Emirati residential exterior with wind tower, coral stone facade",
        "Emirati heritage house exterior, gypsum plaster walls, wooden door, arched windows",
        "modern Emirati villa exterior, fusion of heritage and contemporary design",
        "contemporary Emirati house, flat roof, large glazing, mashrabiya-inspired screens",
        "Emirati heritage majlis interior, carpets, low seating, carved wooden details",
        "modern Emirati living room interior, neutral palette, heritage motifs",
        "close-up of an Emirati mashrabiya wooden screen, intricate geometric pattern",
        "close-up of a barjeel wind tower, four sides, wooden support beams, coral stone",
    ]
    TEST_PROMPTS = (TEST_PROMPTS * 13)[:100]
if QUICK:
    TEST_PROMPTS = TEST_PROMPTS[:25]
SEEDS = [CFG.seed + i for i in range(1 if QUICK else CFG.images_per_prompt)]

import os, glob
def find_real_images(root):
    exts = ("jpg","jpeg","JPG","JPEG","png","PNG")
    paths = []
    for e in exts:
        paths += glob.glob(os.path.join(root, "**", f"*.{e}"), recursive=True)
    # CHDB_Full holds NAME.jpg + NAME-masklabel.png together — exclude the masks
    paths = [p for p in paths if "-masklabel" not in os.path.basename(p).lower()]
    return sorted(set(paths))

print("drive_folder exists:", os.path.isdir(CFG.drive_folder))
REAL_PATHS = find_real_images(CFG.drive_folder)
print("real images found:", len(REAL_PATHS))
if REAL_PATHS[:3]: print("e.g.", [os.path.basename(p) for p in REAL_PATHS[:3]])
assert REAL_PATHS, f"No images under {CFG.drive_folder} — check the path is correct and Drive is mounted."

drive_folder exists: True
real images found: 634
e.g. ['Abdul Raheem Jasim house (north-east elevation).JPG', 'Al Aboudi house (West elevation 1).JPG', 'Al Aboudi house (West elevation 2).JPG']


In [ ]:
# --- Pipeline + metric helpers ---
import numpy as np
from PIL import Image
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from torchmetrics.multimodal.clip_score import CLIPScore
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchvision import transforms as T

_BASE_PIPE = {"p": None}
def load_pipeline(lora_ckpt=None):
    if _BASE_PIPE["p"] is None:
        vae = AutoencoderKL.from_pretrained(CFG.vae_model, torch_dtype=DTYPE)
        pipe = StableDiffusionXLPipeline.from_pretrained(CFG.base_model, vae=vae, torch_dtype=DTYPE)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        pipe.to(DEVICE); pipe.set_progress_bar_config(disable=True)
        _BASE_PIPE["p"] = pipe
    pipe = _BASE_PIPE["p"]
    pipe.unload_lora_weights()
    if lora_ckpt:
        pipe.load_lora_weights(lora_ckpt)
    return pipe

def generate(pipe, tag):
    out = os.path.join(GEN_ROOT, tag); os.makedirs(out, exist_ok=True)
    recs = []
    for pi, prompt in enumerate(TEST_PROMPTS):
        for si, sd in enumerate(SEEDS):
            fp = os.path.join(out, f"{pi:03d}_{si}.png")
            if not os.path.exists(fp):
                g = torch.Generator(device=DEVICE).manual_seed(sd)
                img = pipe(prompt=prompt, num_inference_steps=CFG.eval_steps,
                           guidance_scale=CFG.guidance, generator=g,
                           height=CFG.resolution, width=CFG.resolution).images[0]
                img.save(fp)
            recs.append({"path": fp, "prompt": prompt})
    return recs

to_uint8 = T.Compose([T.Resize((299,299)), T.ToTensor()])
def load_uint8(p): return (to_uint8(Image.open(p).convert("RGB"))*255).to(torch.uint8)
_REAL = {"s": None}
def real_uint8():
    if _REAL["s"] is None:
        _REAL["s"] = torch.stack([load_uint8(p) for p in REAL_PATHS])
    return _REAL["s"]

def clip_scores(gen):
    m = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(DEVICE)
    v = []
    for g in gen:
        img = (T.ToTensor()(Image.open(g["path"]).convert("RGB"))*255).to(torch.uint8).to(DEVICE)
        v.append(m(img, g["prompt"]).item())
    return np.array(v)

def fid_against(gen):
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)
    R = real_uint8()
    for i in range(0, len(R), 64): fid.update(R[i:i+64].to(DEVICE), real=True)
    for i in range(0, len(gen), 64):
        fid.update(torch.stack([load_uint8(g["path"]) for g in gen[i:i+64]]).to(DEVICE), real=False)
    return fid.compute().item()

def lpips_ssim_vs_base(gen, base):
    lp = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True).to(DEVICE)
    ss = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)
    tf = T.Compose([T.Resize((512,512)), T.ToTensor()])
    lv, sv = [], []
    for g, b in zip(gen, base):
        a = tf(Image.open(g["path"]).convert("RGB")).unsqueeze(0).to(DEVICE)
        c = tf(Image.open(b["path"]).convert("RGB")).unsqueeze(0).to(DEVICE)
        lv.append(lp(a, c).item()); sv.append(ss(a, c).item())
    return np.array(lv), np.array(sv)
print("helpers ready")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


helpers ready


In [ ]:
# --- robust CLIP score via open_clip (independent of the broken transformers CLIP API) ---
!pip -q install open_clip_torch
import open_clip, torch, numpy as np
from PIL import Image

_OC = {"m": None}
def _oc():
    if _OC["m"] is None:
        model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
        tok = open_clip.get_tokenizer("ViT-B-32")
        _OC["m"] = (model.eval().to(DEVICE), preprocess, tok)
    return _OC["m"]

@torch.no_grad()
def clip_scores(gen):
    model, preprocess, tok = _oc()
    vals = []
    for g in gen:
        img = preprocess(Image.open(g["path"]).convert("RGB")).unsqueeze(0).to(DEVICE)
        txt = tok([g["prompt"]]).to(DEVICE)                 # 77-token context, truncates
        imf = model.encode_image(img); txf = model.encode_text(txt)
        imf = imf / imf.norm(dim=-1, keepdim=True)
        txf = txf / txf.norm(dim=-1, keepdim=True)
        vals.append((100.0 * (imf * txf).sum(-1)).clamp(min=0).item())
    return np.array(vals)
print("clip_scores now uses open_clip ViT-B-32 (openai)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
clip_scores now uses open_clip ViT-B-32 (openai)


In [ ]:
# --- Discover checkpoints (baseline + every ablation) ---
import glob, os
ckpt_dirs = sorted(glob.glob(os.path.join(CFG.output_dir, "lora_*_best")))
MODELS = {}
for d in ckpt_dirs:
    name = os.path.basename(d)[len("lora_"):-len("_best")]
    MODELS["baseline" if name == "default" else name] = d
print("found", len(MODELS), "checkpoints:")
for k, v in MODELS.items(): print(f"  {k:16s} <- {v}")
assert MODELS, "No lora_*_best checkpoints found in output_dir."

found 8 checkpoints:
  baseline         <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_default_best
  half_data        <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_half_data_best
  no_masked_loss   <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_no_masked_loss_best
  no_snr           <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_no_snr_best
  opt_adamw8bit    <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_opt_adamw8bit_best
  rank16           <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_rank16_best
  rank64           <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_rank64_best
  sched_cosine     <- /content/drive/MyDrive/TurathiAI Revision/Training/outputs/lora_sched_cosine_best


In [ ]:
!pip -q uninstall -y torchao

In [ ]:
# --- Generate + score every model (base zero-shot first, for LPIPS/SSIM reference) ---
import pandas as pd
from IPython.display import display
base_pipe = load_pipeline(lora_ckpt=None)
GEN_BASE = generate(base_pipe, "base")
clip_base = clip_scores(GEN_BASE); fid_base = fid_against(GEN_BASE)

per_image_clip = {"base": clip_base}
rows = [{"model":"base (zero-shot)","CLIP_mean":clip_base.mean(),"CLIP_sd":clip_base.std(),
         "FID_vs_real":fid_base,"LPIPS_vs_base":0.0,"SSIM_vs_base":1.0}]

for name, ckpt in MODELS.items():
    pipe = load_pipeline(lora_ckpt=ckpt)
    gen = generate(pipe, name)
    c = clip_scores(gen); f = fid_against(gen)
    lp, ss = lpips_ssim_vs_base(gen, GEN_BASE)
    per_image_clip[name] = c
    rows.append({"model":name,"CLIP_mean":c.mean(),"CLIP_sd":c.std(),
                 "FID_vs_real":f,"LPIPS_vs_base":lp.mean(),"SSIM_vs_base":ss.mean()})
    print(f"{name:16s} CLIP {c.mean():.3f} | FID {f:.2f} | LPIPS {lp.mean():.3f} | SSIM {ss.mean():.3f}")

iqa = pd.DataFrame(rows).round(4)
display(iqa)
iqa.to_csv(os.path.join(CFG.output_dir, "ablation_iqa.csv"), index=False)
pd.DataFrame(per_image_clip).to_csv(os.path.join(CFG.output_dir, "clip_per_image_all.csv"), index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

...

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


rank64           CLIP 31.207 | FID 130.84 | LPIPS 0.601 | SSIM 0.552


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


sched_cosine     CLIP 30.709 | FID 127.23 | LPIPS 0.634 | SSIM 0.538


,model,CLIP_mean,CLIP_sd,FID_vs_real,LPIPS_vs_base,SSIM_vs_base
0,base (zero-shot),31.5281,1.8919,152.2337,0.0000,1.0000
1,baseline,30.9708,1.8541,129.1873,0.5859,0.5672
2,half_data,31.3453,1.6766,134.1765,0.5804,0.5771
3,no_masked_loss,30.7919,1.7598,135.9742,0.6034,0.5433
4,no_snr,30.9347,1.9183,130.0348,0.5885,0.5649
5,opt_adamw8bit,30.7646,1.7633,123.5719,0.6261,0.5384
6,rank16,30.8901,1.8509,126.4658,0.6077,0.5660
7,rank64,31.2067,1.8056,130.8404,0.6014,0.5517
8,sched_cosine,30.7092,1.7303,127.2301,0.6339,0.5378


In [ ]:
# --- Omnibus ANOVA + eta^2, Friedman + Kendall's W, Holm pairwise (R1-C6) ---
import json, numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

names  = list(per_image_clip.keys())
groups = [per_image_clip[n] for n in names]

F, p = stats.f_oneway(*groups)
grand = np.concatenate(groups); gm = grand.mean()
ss_between = sum(len(g)*(g.mean()-gm)**2 for g in groups)
ss_total   = ((grand-gm)**2).sum()
eta2 = float(ss_between/ss_total)

minlen = min(len(g) for g in groups); aligned = [g[:minlen] for g in groups]
chi, pf = stats.friedmanchisquare(*aligned)
W = float(chi/(minlen*(len(groups)-1)))

omni = {"models": names, "n_per_model": [len(g) for g in groups],
        "anova_F": float(F), "anova_p": float(p), "eta2": eta2,
        "friedman_chi2": float(chi), "friedman_p": float(pf), "kendalls_W": W}
print("OMNIBUS ANOVA: F=%.3f p=%.4g eta2=%.3f" % (F, p, eta2))
print("FRIEDMAN: chi2=%.3f p=%.4g Kendall's W=%.3f" % (chi, pf, W))
json.dump(omni, open(os.path.join(CFG.output_dir, "anova_omnibus.json"), "w"), indent=2)

import pandas as pd
pairs, pvals, recs = [], [], []
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = aligned[i], aligned[j]
        t, pp = stats.ttest_rel(a, b)
        d = (a-b).mean()/(a-b).std(ddof=1)
        recs.append({"model_a":names[i],"model_b":names[j],"t":t,"p_raw":pp,"cohens_d":d})
        pvals.append(pp)
rej, p_holm, *_ = multipletests(pvals, method="holm")
for r, ph, rj in zip(recs, p_holm, rej): r["p_holm"]=ph; r["significant"]=bool(rj)
pw = pd.DataFrame(recs).round(4)
display(pw)
pw.to_csv(os.path.join(CFG.output_dir, "pairwise_stats.csv"), index=False)

OMNIBUS ANOVA: F=9.760 p=1.702e-13 eta2=0.021
FRIEDMAN: chi2=123.949 p=5.065e-23 Kendall's W=0.039


,model_a,model_b,t,p_raw,cohens_d,p_holm,significant
0,base,baseline,5.8275,0.0000,0.2914,0.0000,True
1,base,half_data,2.2441,0.0254,0.1122,0.3552,False
2,base,no_masked_loss,7.5432,0.0000,0.3772,0.0000,True
3,base,no_snr,6.1370,0.0000,0.3069,0.0000,True
4,base,opt_adamw8bit,7.7892,0.0000,0.3895,0.0000,True
5,base,rank16,6.7737,0.0000,0.3387,0.0000,True
6,base,rank64,3.6152,0.0003,0.1808,0.0068,True
7,base,sched_cosine,8.2851,0.0000,0.4143,0.0000,True
8,baseline,half_data,-4.3373,0.0000,-0.2169,0.0004,True
9,baseline,no_masked_loss,1.9269,0.0547,0.0963,0.6018,False


In [ ]:
# --- Merge val-loss ablation table with IQA -> manuscript-ready Table 23 ---
import os, pandas as pd
val_csv = os.path.join(CFG.output_dir, "ablation_val.csv")
if os.path.exists(val_csv):
    val = pd.read_csv(val_csv)[["ablation","best_val"]]
    iqa2 = iqa.rename(columns={"model":"ablation"}).copy()
    merged = iqa2.merge(val, on="ablation", how="left").round(4)
    display(merged)
    merged.to_csv(os.path.join(CFG.output_dir, "ablation_table_full.csv"), index=False)
    print("\nSaved ablation_table_full.csv -> drop straight into manuscript Table 23.")
else:
    print("ablation_val.csv not found; ablation_iqa.csv already holds the IQA table.")

,ablation,CLIP_mean,CLIP_sd,FID_vs_real,LPIPS_vs_base,SSIM_vs_base,best_val
0,base (zero-shot),31.5281,1.8919,152.2337,0.0000,1.0000,NaN
1,baseline,30.9708,1.8541,129.1873,0.5859,0.5672,NaN
2,half_data,31.3453,1.6766,134.1765,0.5804,0.5771,0.0608
3,no_masked_loss,30.7919,1.7598,135.9742,0.6034,0.5433,0.0735
4,no_snr,30.9347,1.9183,130.0348,0.5885,0.5649,0.0945
5,opt_adamw8bit,30.7646,1.7633,123.5719,0.6261,0.5384,0.0607
6,rank16,30.8901,1.8509,126.4658,0.6077,0.5660,0.0607
7,rank64,31.2067,1.8056,130.8404,0.6014,0.5517,0.0607
8,sched_cosine,30.7092,1.7303,127.2301,0.6339,0.5378,0.0607



Saved ablation_table_full.csv -> drop straight into manuscript Table 23.


## Interpreting the outputs

- **`anova_omnibus.json`** — answers R1-C6 at the omnibus level. Report `anova_F`, `anova_p`, and **`eta2`** together; eta-squared is the share of CLIP variance explained by model identity. A significant `p` with a small eta-squared (~<0.06) is the honest "statistically detectable but small" story.
- **`pairwise_stats.csv`** — Holm-corrected paired tests with **Cohen's d** for every model pair; use it to state which differences are both significant *and* non-trivial.
- **`ablation_iqa.csv` / `ablation_table_full.csv`** — extends the val-loss ablation (Table 23) with CLIP / FID / LPIPS / SSIM per configuration, so rank/optimizer/scheduler effects are quantified on output quality, not just loss.

**Caveats to keep:** CLIP truncates text at 77 tokens; FID needs the full real set as reference (state `n`); generations are seed-aligned across models so Friedman/Kendall's W are valid. Run `QUICK=False` for the paper-scale numbers — `QUICK=True` is only a wiring check.